# DX 704 Week 5 Project

This week's project will test your understanding of this week's concepts by asking you to simulate various algorithms by hand.
You will apply search, minimax and dynamic programming concepts to solve a variety of small planning problems.

The full project description, a template notebook and supporting materials are available on GitHub: [Project 5 Materials](https://github.com/bu-cds-dx704/dx704-project-05).

## Example Code

This week's assignment does not involve any coding.

## Part 1: Searching vs Games

Consider the state space illustrated below.
Each terminal state state is marked with a reward for reaching that state.
Each non-terminal state has two possible actions represented by the two outgoing arrows to later (lower) states.
The only rewards are for reaching the terminal states, there are no diminishing returns (i.e. $\gamma=1$), and there is no randomness so actions may be freely chosen.

![](https://github.com/bu-cds-dx704/dx704-project-05/blob/main/part1.png?raw=true)

Solve for the value of each non-terminal state according to the following three scenarios.

1. Search: There is one agent that picks all actions with the goal of maximizing the final reward.
2. Minimax: There are two agents P1 and P2. P1 controls the actions for the 1st and 3rd rows (i.e. the states marked A and D-G) while P2 controls the actions for the 2nd and 4th rows (i.e. the states B-C and H-J). P1 seeks to maximize the final reward, and P2 seeks to minimize the final reward.
3. Maximin: P1 and P2 control the same states as before, but P1 seeks to minimize the final reward, and P2 seeks to maximize the final reward.

Save your results in a file "values-1.tsv" with the column state with label A-J and columns search_value, minimax_value, and maximin_value that respectively correspond to the three scenarios.

In [ ]:
import pandas as pd

# Terminal rewards / children of each non-terminal state
tree = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["H", "I"],
    "E": ["J", 5],
    "F": [1, -1],
    "G": [1, 0],
    "H": [9, -1],
    "I": [20, 1],
    "J": [3, 2]
}

# Work from the bottom of the tree upward
order = ["H", "I", "J", "D", "E", "F", "G", "B", "C", "A"]


def solve(mode):
    values = {}

    for state in order:
        children = tree[state]

        # Get the value of each child
        child_values = [
            values[x] if isinstance(x, str) else x
            for x in children
        ]

        if mode == "search":
            # One agent always maximizes
            values[state] = max(child_values)

        elif mode == "minimax":
            # P1 controls A and D-G and maximizes
            # P2 controls B-C and H-J and minimizes
            if state in ["A", "D", "E", "F", "G"]:
                values[state] = max(child_values)
            else:
                values[state] = min(child_values)

        elif mode == "maximin":
            # P1 controls A and D-G and minimizes
            # P2 controls B-C and H-J and maximizes
            if state in ["A", "D", "E", "F", "G"]:
                values[state] = min(child_values)
            else:
                values[state] = max(child_values)

    return values


# Solve all three scenarios
search = solve("search")
minimax = solve("minimax")
maximin = solve("maximin")


# Create required output
states = list("ABCDEFGHIJ")

results = pd.DataFrame({
    "state": states,
    "search_value": [search[s] for s in states],
    "minimax_value": [minimax[s] for s in states],
    "maximin_value": [maximin[s] for s in states]
})

results.to_csv("values-1.tsv", sep="\t", index=False)

print(results)

  state  search_value  minimax_value  maximin_value
0     A            20              1              0
1     B            20              1              9
2     C             1              1              0
3     D            20              1              9
4     E             5              5              3
5     F             1              1             -1
6     G             1              1              0
7     H             9             -1              9
8     I            20              1             20
9     J             3              2              3


Hint: Print out the image above and compute the values by hand in a bottom up fashion.

Submit the file "values-1.tsv" in Gradescope.

## Part 2: Picking Up Sticks

The state space illustrated below corresponds to a variation of the game [Nim](https://en.wikipedia.org/wiki/Nim).
States labeled with a prefix of "p1_" correspond to states where player P1 chooses the action while states labeled with a prefix of "p2_" correspond to states where player P2 chooses the action.
The number in the suffix is the number of "sticks" remaining.
The players take turns choosing actions, and each action corresponds to removing one or two sticks.
When there are no more sticks, the player who would have picked an action loses.


![](https://github.com/bu-cds-dx704/dx704-project-05/blob/main/part2.png?raw=true)

For example, from the state labeled "p1_1", there is one stick left, player P1 removes the last stick, and player P2 loses.
The loss for P2 is represented by a final reward of +1.
A loss for P1 is represented by a final reward of -1.
Player P1 tries to maximize the final reward, and player P2 tries to minimize the final reward.

In [ ]:


# Terminal states
# If it is P1's turn with 0 sticks, P1 loses -> -1
# If it is P2's turn with 0 sticks, P2 loses -> +1
values = {
    "p1_0": -1,
    "p2_0": 1
}

# Solve from 1 stick up to 5 sticks
for sticks in range(1, 6):

    # P1 can remove 1 or 2 sticks.
    # P1 wants to MAXIMIZE the final reward.
    p1_options = []

    for remove in [1, 2]:
        if sticks - remove >= 0:
            next_state = f"p2_{sticks - remove}"
            p1_options.append(values[next_state])

    values[f"p1_{sticks}"] = max(p1_options)

    # P2 can remove 1 or 2 sticks.
    # P2 wants to MINIMIZE the final reward.
    p2_options = []

    for remove in [1, 2]:
        if sticks - remove >= 0:
            next_state = f"p1_{sticks - remove}"
            p2_options.append(values[next_state])

    values[f"p2_{sticks}"] = min(p2_options)


# Non-terminal states only
states = []

for sticks in range(1, 6):
    states.append(f"p1_{sticks}")
    states.append(f"p2_{sticks}")


results = pd.DataFrame({
    "state": states,
    "value": [values[state] for state in states]
})

results.to_csv("values-2.tsv", sep="\t", index=False)

print(results)

  state  value
0  p1_1      1
1  p2_1     -1
2  p1_2      1
3  p2_2     -1
4  p1_3     -1
5  p2_3      1
6  p1_4      1
7  p2_4     -1
8  p1_5      1
9  p2_5     -1


Submit the file "values-2.tsv" in Gradescope.

## Part 3: Searching a Maze

Consider the following maze.

![](https://github.com/bu-cds-dx704/dx704-project-05/blob/main/part3.png?raw=true)

State C is a terminal state giving reward +100.
The remaining states have a reward of -1 when they are reached.
So moving to state F has a value of +99 due to the reward of -1 at state F and the optimal action of moving to state C for the reward of +100 afterwards.

Compute the values for states A-J and S and save them in a file "values-3.tsv" with columns state and value.

In [ ]:


# State transitions
transitions = {
    "A": ["S"],
    "B": ["D", "S"],
    "D": ["B", "E", "G"],
    "E": ["D", "H"],
    "F": ["C", "J"],
    "G": ["D", "I"],
    "H": ["E", "I"],
    "I": ["G", "H", "J"],
    "J": ["F", "I"],
    "S": ["A", "B"]
}

# C is terminal with reward +100
values = {
    "A": 0,
    "B": 0,
    "C": 100,
    "D": 0,
    "E": 0,
    "F": 0,
    "G": 0,
    "H": 0,
    "I": 0,
    "J": 0,
    "S": 0
}

# Value iteration
while True:
    new_values = values.copy()

    for state, next_states in transitions.items():
        # Reaching any non-terminal next state costs -1
        new_values[state] = max(
            -1 + values[next_state]
            for next_state in next_states
        )

    # Terminal state stays fixed
    new_values["C"] = 100

    # Stop when values no longer change
    if all(new_values[s] == values[s] for s in values):
        values = new_values
        break

    values = new_values


# Required state order
states = list("ABCDEFGHIJ") + ["S"]

results = pd.DataFrame({
    "state": states,
    "value": [values[state] for state in states]
})

results.to_csv("values-3.tsv", sep="\t", index=False)

print(results)

   state  value
0      A     92
1      B     94
2      C    100
3      D     95
4      E     95
5      F     99
6      G     96
7      H     96
8      I     97
9      J     98
10     S     93


Submit "values-3.tsv" in Gradescope.

## Part 4: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

Yes I have used AI to write the code since no example was given and to understand the questions connections to the lessons we learned this week. I had to input the dot files provided for AI to organize the steps one by one for my comprehension and to optimize the required code.